In [7]:
from kafka import KafkaProducer, KafkaConsumer
from typing import List
import json
import psycopg2
from psycopg2.extras import execute_values
from confluent_kafka import Consumer, KafkaException
import json
import signal
import sys

In [12]:
def kafka_consumer():

	conf = {
		'bootstrap.servers': 'localhost:9092',
		'group.id': 'review-group',
		'auto.offset.reset': 'earliest',  # earliest | latest
		'enable.auto.commit': False
	}

	consumer = Consumer(conf)

	topic = "reviews"
	consumer.subscribe([topic])
	empty_polls = 0
	max_empty_polls = 5  # para após 5 tentativas sem mensagem

	try:
		json_records = []
		records = consumer.consume(timeout=1.0, num_messages=100)

		if records is None:
			empty_polls += 1
			print(f"Sem mensagens... ({empty_polls})")

			if empty_polls >= max_empty_polls:
				print("Não há mais mensagens. Encerrando.")
				return []
		
		for msg in records:
			if msg.error():
				raise KafkaException(msg.error())
			else:

				# Caso a mensagem esteja em JSON
				try:
					value = json.loads(msg.value().decode('utf-8'))
				except Exception:
					value = msg.value().decode('utf-8')

				json_records.append(value)
				consumer.commit(msg)

		return json_records

	finally:
		consumer.close()
		print("Consumer encerrado com sucesso.")


def write_batch_to_postgres(records: list, table: str, conn_params: dict):
    """Write a list of JSON-serializable records to Postgres in a specified table.

    If the table does not exist, it will be created with a simple schema
    (`id SERIAL PRIMARY KEY, data JSONB`). Records are inserted as JSONB.
    """
    conn = psycopg2.connect(**conn_params)
    cur = conn.cursor()
    cur.execute(
        f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id SERIAL PRIMARY KEY,
            product_name varchar,
			product_category varchar,
			review_size varchar,
			type varchar,
			content TEXT[],
			review varchar,
			stars int
        )
        """
    )
    conn.commit()

    if records:
        execute_values(
            cur,
            f"INSERT INTO {table} (product_name, product_category, review_size, type, content, review, stars) VALUES %s",
            [
				(
					rec.get("product_name", ""),
					rec.get("product_category", ""),
					rec.get("review_size", ""), 
					rec.get("type", ""), 
					rec.get("content", []), 
					rec.get("review", ""), 
					rec.get("stars", 0)
				) 
				for rec in records
			],
        )
        conn.commit()
    cur.close()
    conn.close()


def kafka_to_postgres(
    topic: str,
    conn_params: dict,
    table: str = "reviews",
    bootstrap_servers: List[str] = ["localhost:9092"],
    batch_size: int = 100,
):
    """Consume messages from Kafka in micro‑batches and load them into Postgres.

    The function repeatedly reads up to `batch_size` messages using
    ``kafka_consumer`` and writes each batch to the specified Postgres table
    until there are no more available messages. ``conn_params`` should be a
    dict suitable for ``psycopg2.connect`` (e.g. ``{
    "host":...,"port":...,"user":...,"password":...,"dbname":...}``).
    """
    while True:
        batch = kafka_consumer()#(topic, bootstrap_servers=bootstrap_servers, max_messages=batch_size)
        if not batch:
            print("no new messages available, stopping consumer")
            break
        write_batch_to_postgres(batch, table, conn_params)
        print(f"inserted microbatch of {len(batch)} records into {table}")


In [13]:
# example usage
conn_params = {
    "host": "localhost",
    "port": 5432,
    "user": "dev_user",
    "password": "1234",
    # the default postgres database is 'postgres'; set to your target DB if different
    "dbname": "reviews_dev",
}

# consume all available reviews in microbatch sizes of 100 and load into Postgres
default_batch = 100
kafka_to_postgres("reviews", conn_params, table="reviews", batch_size=default_batch)


Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted microbatch of 100 records into reviews
Consumer encerrado com sucesso.
inserted